In [1]:
import numpy as np
import cv2
import pickle
import os

# Load Phase 2 results
with open('phase2_results.pkl', 'rb') as f:
    p2 = pickle.load(f)

img = p2['img']
height, width = p2['height'], p2['width']
region_map = p2['region_map']
special_regions = p2['special_regions']

output_dir = "comic/phase3_output"
os.makedirs(output_dir, exist_ok=True)

print(f"Phase 2 data loaded")
print(f"Special regions (text+edge): {len(special_regions)}")

# Get gradient candidate regions (not special)
gradient_candidates = set(np.unique(region_map[region_map >= 0])) - special_regions
print(f"Gradient candidate regions: {len(gradient_candidates)}")

Phase 2 data loaded
Special regions (text+edge): 11387
Gradient candidate regions: 2271


In [2]:
# Gradient detection for Phase 3: CGV-style derivative histogram (Chang & Su, 2015)
# or a least-squares plane fit, selectable below.
METHOD = 'cgv'            # 'cgv' = derivative-direction histogram, 'plane' = least-squares plane fit
USE_DENOISE = True        # bilateral filter before the analysis (first step of CGV)

MIN_PIXELS = 50           # smaller regions are never treated as gradients
MIN_RANGE = 12.0          # minimum total colour change (gray levels) along the ramp
MAG_EPS = 0.05            # pixels with a weaker slope (levels/pixel) have no usable direction
DOMINANT_FRACTION = 0.4   # cgv: share of pixels that must agree on the dominant direction
MAX_RESIDUAL = 0.5        # plane: residual must stay below this fraction of the flat variance

# --- Pre-processing: bilateral filter and the 7x7 derivative masks from the CGV paper ---
analysis_img = cv2.bilateralFilter(img, 9, 30, 9) if USE_DENOISE else img
gray_a = cv2.cvtColor(analysis_img, cv2.COLOR_RGB2GRAY).astype(np.float64)

row = np.array([-1, -2, -3, 0, 3, 2, 1], dtype=np.float64)
Mx = np.tile(row, (7, 1))     # horizontal derivative mask
My = Mx.T.copy()              # vertical derivative mask
Gx = cv2.filter2D(gray_a, cv2.CV_64F, Mx, borderType=cv2.BORDER_REPLICATE)
Gy = cv2.filter2D(gray_a, cv2.CV_64F, My, borderType=cv2.BORDER_REPLICATE)

# A ramp with slope s levels/pixel gives |G| = 140 * s with these masks
grad_mag = np.hypot(Gx, Gy) / 140.0
grad_ang = np.degrees(np.arctan2(Gy, Gx)) % 360   # angle in (x, y) image coordinates

def gradient_endpoints(pixels_yx, direction, img, frac=0.05):
    """
    Project pixels onto the gradient direction and return the mean colour
    of the lowest and highest `frac` of the projection (start/end colours).
    Pixels are converted to (x, y) order to match the decoder convention.
    """
    xy = pixels_yx[:, ::-1].astype(float)
    angle = np.radians(direction * 45)
    proj = xy @ np.array([np.cos(angle), np.sin(angle)])
    order = np.argsort(proj)
    k = max(1, int(len(order) * frac))
    lo = pixels_yx[order[:k]]
    hi = pixels_yx[order[-k:]]
    start = img[lo[:, 0], lo[:, 1]].astype(float).mean(axis=0)
    end = img[hi[:, 0], hi[:, 1]].astype(float).mean(axis=0)
    return start, end

def region_interior(region_mask, erode_px=2):
    """Erode the mask so that outlines at the region border do not disturb the analysis."""
    inner = cv2.erode(region_mask.astype(np.uint8), np.ones((3, 3), np.uint8),
                      iterations=erode_px).astype(bool)
    return inner if inner.sum() >= MIN_PIXELS else region_mask

def ramp_extent(xs, ys, direction):
    angle = np.radians(direction * 45)
    proj = xs * np.cos(angle) + ys * np.sin(angle)
    return proj.max() - proj.min()

def cgv_analysis(region_mask):
    """Direction histogram of the derivative angles; returns (direction, share, range) or None."""
    inner = region_interior(region_mask)
    ys, xs = np.nonzero(inner)
    mag, ang = grad_mag[ys, xs], grad_ang[ys, xs]
    valid = mag > MAG_EPS
    if valid.sum() < MIN_PIXELS // 2:
        return None
    bins = np.round(ang[valid] / 45).astype(int) % 8          # bins centred on multiples of 45 deg
    hist = np.bincount(bins, minlength=8) / valid.sum()
    direction = int(np.argmax(hist))
    slope = float(np.median(mag))                             # typical levels per pixel
    return direction, float(hist[direction]), slope * ramp_extent(xs, ys, direction)

def plane_analysis(region_mask):
    """Least-squares plane colour = a*x + b*y + c; returns (direction, range, residual ratio)."""
    inner = region_interior(region_mask)
    ys, xs = np.nonzero(inner)
    A = np.column_stack([xs, ys, np.ones(len(xs))]).astype(float)
    V = analysis_img[ys, xs].astype(float)
    coef, *_ = np.linalg.lstsq(A, V, rcond=None)
    resid = V - A @ coef
    mse_plane = np.mean(resid ** 2, axis=0).sum()
    mse_flat = V.var(axis=0).sum()
    mag = np.hypot(coef[0], coef[1])
    k = int(np.argmax(mag))
    angle = np.arctan2(coef[1][k], coef[0][k])
    direction = int(np.round(angle / (np.pi / 4))) % 8
    proj = xs * np.cos(angle) + ys * np.sin(angle)
    return direction, mag[k] * (proj.max() - proj.min()), mse_plane / max(mse_flat, 1e-9)

def detect_gradient(rid, rmap, img):
    region_mask = (rmap == rid)
    pixels_yx = np.argwhere(region_mask)
    if len(pixels_yx) < MIN_PIXELS:
        return 'flat'

    if METHOD == 'cgv':
        res = cgv_analysis(region_mask)
        if res is None:
            return 'flat'
        direction, share, total_range = res
        if share < DOMINANT_FRACTION or total_range < MIN_RANGE:
            return 'flat'
    else:
        direction, total_range, residual = plane_analysis(region_mask)
        if total_range < MIN_RANGE or residual > MAX_RESIDUAL:
            return 'flat'

    start_color, end_color = gradient_endpoints(pixels_yx, direction, img)
    return ('gradient', direction, start_color, end_color)

# Classify regions
gradient_regions = {}
flat_regions = []

for rid in gradient_candidates:
    result = detect_gradient(rid, region_map, img)
    if result == 'flat':
        flat_regions.append(rid)
    else:
        gradient_regions[rid] = {
            'direction': result[1],
            'start_color': result[2],
            'end_color': result[3]
        }

print(f"\nGradient detection results ({METHOD}, denoise={USE_DENOISE}):")
print(f"  Gradient regions: {len(gradient_regions)}")
print(f"  Flat regions: {len(flat_regions)}")


Gradient detection results (cgv, denoise=True):
  Gradient regions: 56
  Flat regions: 2215


In [3]:
def get_direct_neighbors(rid, rmap):
    mask = (rmap == rid).astype(np.uint8)
    # A 3x3 cross dilation does not wrap around the image borders (np.roll did)
    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
    ring = cv2.dilate(mask, kernel).astype(bool) & ~mask.astype(bool)
    ids = np.unique(rmap[ring])
    return [int(i) for i in ids if i != rid and i != -1]

def directions_compatible(dir1, dir2, tolerance=1):
    diff = abs(dir1 - dir2)
    diff = min(diff, 8 - diff)
    return diff <= tolerance

def boundary_color_jump(rid, nid, rmap, img):
    """Mean colour jump across the shared boundary of two adjacent regions."""
    a = (rmap == rid).astype(np.uint8)
    b = (rmap == nid).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
    touch_b = cv2.dilate(a, kernel).astype(bool) & b.astype(bool)  # pixels of b next to a
    touch_a = cv2.dilate(b, kernel).astype(bool) & a.astype(bool)  # pixels of a next to b
    if not touch_a.any() or not touch_b.any():
        return float('inf')
    ca = img[touch_a].astype(float).mean(axis=0)
    cb = img[touch_b].astype(float).mean(axis=0)
    return float(np.linalg.norm(ca - cb))

# Two adjacent gradient regions are merged only if the colour is continuous
# across their shared boundary (a true gradient has no visible step there)
MAX_BOUNDARY_JUMP = 12.0

# Union-Find (reset on every run so stale entries cannot leak in)
parent = {}

def find(x):
    if x not in parent:
        parent[x] = x
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    px, py = find(x), find(y)
    if px != py:
        parent[px] = py

for rid in gradient_regions.keys():
    for nid in get_direct_neighbors(rid, region_map):
        if nid not in gradient_regions:
            continue
        if not directions_compatible(gradient_regions[rid]['direction'],
                                     gradient_regions[nid]['direction']):
            continue
        if boundary_color_jump(rid, nid, region_map, img) < MAX_BOUNDARY_JUMP:
            union(rid, nid)

groups = {}
for rid in gradient_regions.keys():
    groups.setdefault(find(rid), []).append(rid)

print(f"After merging adjacent gradients: {len(groups)} groups")
sizes = sorted((int(np.isin(region_map, m).sum()) for m in groups.values()), reverse=True)
print(f"Largest groups (pixels): {sizes[:5]}")

# Build one merged gradient region per group
merged_gradient_regions = {}

for root, members in groups.items():
    group_mask = np.isin(region_map, members)
    pixels_yx = np.argwhere(group_mask)
    direction = gradient_regions[root]['direction']
    start_color, end_color = gradient_endpoints(pixels_yx, direction, img)

    # Relabel every member with the root id, so later phases see ONE region
    region_map[group_mask] = root

    merged_gradient_regions[root] = {
        'direction': direction,
        'start_color': start_color,
        'end_color': end_color,
        'merged_from': members
    }

gradient_regions = merged_gradient_regions
print(f"Merged gradient regions: {len(gradient_regions)}")

After merging adjacent gradients: 51 groups
Largest groups (pixels): [25111, 13726, 10401, 9919, 4818]
Merged gradient regions: 51


In [4]:
phase3_results = {
    'img': img,
    'height': height,
    'width': width,
    'region_map': region_map,
    'special_regions': special_regions,
    'gradient_regions': gradient_regions,
    'flat_regions': flat_regions
}

with open('phase3_results.pkl', 'wb') as f:
    pickle.dump(phase3_results, f)

print("Phase 3 results saved to phase3_results.pkl")
print(f"Summary:")
print(f"  Special regions (text+edge): {len(special_regions)}")
print(f"  Gradient regions: {len(gradient_regions)}")
print(f"  Flat regions: {len(flat_regions)}")

Phase 3 results saved to phase3_results.pkl
Summary:
  Special regions (text+edge): 11387
  Gradient regions: 51
  Flat regions: 2215


In [5]:
# Visualize the classification results

# Create visualization
vis = img.copy().astype(float)

# Mark special regions (text+edge) - Red
for rid in special_regions:
    mask = (region_map == rid)
    vis[mask] = [255, 0, 0]  # Red

# Mark gradient regions - Green
for rid in gradient_regions.keys():
    mask = (region_map == rid)
    vis[mask] = [0, 255, 0]  # Green

# Mark flat regions - Blue (just a few for reference)
flat_count = 0
for rid in flat_regions:
    if flat_count < 100:  # Just show first 100 for visibility
        mask = (region_map == rid)
        vis[mask] = [0, 0, 255]  # Blue
        flat_count += 1

# Save visualization
vis_path = os.path.join(output_dir, "01_phase3_classification.jpg")
cv2.imwrite(vis_path, cv2.cvtColor(vis.astype(np.uint8), cv2.COLOR_RGB2BGR))
print(f"Classification visualization saved: {vis_path}")

print(f"\nColor legend:")
print(f"  Red = Special regions (text+edge)")
print(f"  Green = Gradient regions")
print(f"  Blue = Flat regions (sample)")

Classification visualization saved: comic/phase3_output\01_phase3_classification.jpg

Color legend:
  Red = Special regions (text+edge)
  Green = Gradient regions
  Blue = Flat regions (sample)


In [6]:
# Check the background gradient region around (row 330, col 450)
rid = int(region_map[330, 450])
info = gradient_regions.get(rid)
print("region", rid, "->", "gradient" if info else "NOT a gradient region")
if info:
    print("direction bin :", info['direction'], f"(angle {info['direction'] * 45} deg)")
    print("start colour  :", np.round(info['start_color']).astype(int))
    print("end colour    :", np.round(info['end_color']).astype(int))

region 29459 -> gradient
direction bin : 2 (angle 90 deg)
start colour  : [131 183 155]
end colour    : [210 207 170]
